# Lab 11 — Data Lakehouse Roadmap: Bronze → Silver → Gold

This lab builds a **production-grade medallion pipeline** from scratch on RustFS.
Unlike Lab 02 (10-row synthetic demo), here we:

- **Download 54,000 real records** from a public dataset
- **Inject real-world data quality issues** into Bronze (nulls, duplicates, invalid values)
- **Produce a quality report** showing exactly what Silver fixed
- **Partition Silver by cut** — realistic lakehouse file layout
- **Compute 4 Gold KPIs** with a matplotlib analytics dashboard

### The Business Story

> **DiamondDB Analytics** receives raw GIA (Gemological Institute of America)
> certification records from upstream suppliers as CSV dumps. We need to build
> a pricing intelligence platform so buyers can make data-driven decisions.

### A rule we follow strictly in this lab

**Bronze ingestion never transforms or cleans data.** It only moves bytes from
the source into `s3://bronze/` untouched. All cleaning happens later, in Silver.

**No metadata sidecar files.** Every upload — the Bronze CSV, every Silver
Parquet partition, every Gold KPI table — carries its own metadata **in the
same `PUT` request** (via S3 object `Metadata`), instead of writing a second
`.json` file next to the data. One write, one object, self-describing.

### Medallion Architecture

```
                     ┌─────────────────────────────────────────────────────────┐
  [GIA CSV Feed]     │  RustFS  ·  RS(4,2) Erasure Coding  ·  4 Drives         │
       │             │                                                           │
       ▼             │  s3://bronze/                                             │
  ┌─────────┐        │    diamonds/ingestion_date=2024-01-15/                    │
  │  BRONZE │───────▶│      raw.csv  (54K rows, as received, zero cleaning)      │
  │  Raw    │        │      → metadata rides IN the same PUT (S3 Metadata)       │
  └─────────┘        │                                                           │
       │             │  s3://silver/                                             │
       ▼             │    diamonds/cut=Fair/data.parquet   → metadata in PUT     │
  ┌─────────┐        │    diamonds/cut=Good/data.parquet   → metadata in PUT     │
  │  SILVER │───────▶│    diamonds/cut=Ideal/data.parquet  → metadata in PUT     │
  │  Clean  │        │    diamonds/cut=Premium/data.parquet                      │
  └─────────┘        │    diamonds/cut=Very Good/data.parquet                    │
       │             │                                                           │
       ▼             │  s3://gold/                                               │
  ┌─────────┐        │    kpi/price_by_cut_color.parquet   → metadata in PUT     │
  │  GOLD   │───────▶│    kpi/market_segments.parquet      → metadata in PUT     │
  │  KPIs   │        │    kpi/carat_price_bins.parquet                           │
  └─────────┘        │    kpi/quality_matrix.parquet                             │
                     └─────────────────────────────────────────────────────────┘
```


---
## 0 · Setup

Before touching S3, the next code cell will:

1. Import the libraries we need (`pandas`, `pyarrow`, `numpy`, `matplotlib`, `boto3` via `lab_utils`).
2. Create the S3 client that talks to RustFS.
3. Define the three bucket names — `bronze`, `silver`, `gold` — and the key prefixes each layer will use.
4. Define two small helpers, `s3_put_parquet` and `s3_read_parquet`, that write/read Parquet straight to/from S3 in memory (no local disk).

`s3_put_parquet` accepts an optional `metadata` dict — this is how we'll attach metadata **directly on the same PUT request** later on, instead of writing a separate `.json` file for every dataset.

In [ ]:
import io
import json
import sys
import urllib.request
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import pyarrow as pa
from IPython.display import display

sys.path.insert(0, '../scripts')
from lab_utils import get_s3_client

s3 = get_s3_client()

# Medallion buckets (created by docker-compose init container)
BRONZE = 'bronze'
SILVER = 'silver'
GOLD   = 'gold'

INGESTION_DATE = '2024-01-15'
BRONZE_PREFIX  = f'diamonds/ingestion_date={INGESTION_DATE}'
SILVER_PREFIX  = 'diamonds'
GOLD_PREFIX    = 'kpi'


def _stringify_metadata(metadata: dict) -> dict:
    """S3 object metadata values must be strings; encode dicts/lists as JSON."""
    return {
        k: v if isinstance(v, str) else json.dumps(v)
        for k, v in metadata.items()
    }


def s3_put_parquet(df: pd.DataFrame, bucket: str, key: str, metadata: dict = None, **kwargs) -> int:
    """Write a DataFrame as Parquet directly to S3, no local disk.

    If `metadata` is given, it's attached to the SAME PUT request as S3
    object metadata (x-amz-meta-*) — no separate metadata file needed.
    """
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine='pyarrow', **kwargs)
    size = buf.tell()
    buf.seek(0)
    put_kwargs = {'Bucket': bucket, 'Key': key, 'Body': buf}
    if metadata:
        put_kwargs['Metadata'] = _stringify_metadata(metadata)
    s3.put_object(**put_kwargs)
    return size


def s3_read_parquet(bucket: str, key: str) -> pd.DataFrame:
    """Read a Parquet file from S3 into a DataFrame."""
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))


print('✅ Setup complete')
print(f'   pandas {pd.__version__} | pyarrow {pa.__version__}')

---
## 1 · Bronze — Raw Ingestion

Bronze stores data **exactly as received** from the source system — no filtering, no cleaning, no transformation of any kind. It is your permanent audit trail: if a Silver transformation has a bug, you reprocess from Bronze.

**What "ingestion" means in this lab**: one download, then one `PUT`. Nothing in between touches the row values.

The next code cell only downloads the source file and loads it into memory so we can look at it — it does **not** write anything to Bronze yet, and it does not clean or alter a single value.

In [ ]:
# ── Download real data ────────────────────────────────────────────────────────
URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
print('Downloading diamonds dataset from GIA public feed...')
print(f'  URL: {URL}')

with urllib.request.urlopen(URL) as response:
    raw_bytes = response.read()

df_raw = pd.read_csv(io.BytesIO(raw_bytes))
original_rows = len(df_raw)
print(f'✅ Downloaded {original_rows:,} records  |  {len(raw_bytes)/1024:.0f} KB')
print(f'   Columns: {list(df_raw.columns)}')
display(df_raw.head(3))

### Simulating a real-world dirty feed

The public dataset we just downloaded is already clean, but real supplier feeds rarely are. To make this lab realistic, the next cell **stands in for the supplier's export system** — it injects duplicates, nulls, invalid dimensions, negative carats, and inconsistent labels *before* the file ever reaches us.

⚠️ This is not a transformation performed by our pipeline — think of it as writing the messy CSV that a real GIA feed would have sent. Our actual ingestion step (right after this one) will take this file and store it in Bronze completely untouched.

In [ ]:
# ── Inject realistic data quality issues ──────────────────────────────────────
rng = np.random.default_rng(42)
df_dirty = df_raw.copy()

# 1. Duplicate rows — supplier system resent 400 records
n_dupes = 400
dupe_idx = rng.choice(len(df_dirty), size=n_dupes, replace=False)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[dupe_idx]], ignore_index=True)

# 2. Null prices — supplier redacted prices for unreleased stones
null_price_idx = rng.choice(len(df_dirty), size=300, replace=False)
df_dirty.loc[null_price_idx, 'price'] = np.nan

# 3. Invalid dimensions — sensor calibration errors (x/y/z = 0)
zero_dim_idx = rng.choice(len(df_dirty), size=150, replace=False)
df_dirty.loc[zero_dim_idx, ['x', 'y', 'z']] = 0

# 4. Negative carat weights — data entry errors
neg_carat_idx = rng.choice(len(df_dirty), size=50, replace=False)
df_dirty.loc[neg_carat_idx, 'carat'] = -df_dirty.loc[neg_carat_idx, 'carat']

# 5. Inconsistent cut labels — upstream system changed naming mid-export
df_dirty.loc[rng.choice(len(df_dirty), size=200, replace=False), 'cut'] = \
    df_dirty.loc[rng.choice(len(df_dirty), size=200, replace=False), 'cut'].str.lower()

injected_issues = {
    'duplicate_rows': n_dupes,
    'null_prices': int((df_dirty['price'].isna()).sum()),
    'zero_dimensions': int(((df_dirty['x'] == 0) | (df_dirty['y'] == 0) | (df_dirty['z'] == 0)).sum()),
    'negative_carats': int((df_dirty['carat'] < 0).sum()),
    'bad_cut_labels': int(df_dirty['cut'].str.islower().sum()),
}

print('⚠️  Injected quality issues into raw data:')
for issue, count in injected_issues.items():
    print(f'   {issue:<22}: {count:,}')
print(f'   Total rows (with dupes): {len(df_dirty):,}')

### Bronze ingestion: one PUT, metadata included

Now for the actual ingestion. The next cell will:

1. Serialize the (already-dirty) DataFrame to CSV bytes — this is just serialization, not a transformation of values.
2. Build a small metadata dict describing the ingestion (source URL, ingestion date, row count, columns, the quality issues we know are already in the file).
3. Send **one single `put_object` call** with `Body=csv_bytes` and `Metadata=metadata` — the metadata rides along with the same request that writes the data. No `metadata.json` sidecar file is created.
4. Read the metadata back with `head_object` to prove it's really attached to the object, not just printed to the console.

In [ ]:
# ── Bronze ingestion: one PUT, data + metadata together ───────────────────────
csv_bytes = df_dirty.to_csv(index=False).encode()
csv_key   = f'{BRONZE_PREFIX}/raw.csv'

metadata = {
    'source':         'GIA Certification Feed',
    'source_url':     URL,
    'ingestion_date': INGESTION_DATE,
    'rows_received':  str(len(df_dirty)),
    'columns':        ','.join(df_dirty.columns),
    'known_issues':   injected_issues,   # dict -> JSON string via _stringify_metadata
    'schema_version': '1.0',
}

s3.put_object(
    Bucket=BRONZE,
    Key=csv_key,
    Body=csv_bytes,
    Metadata=_stringify_metadata(metadata),
)

print('✅ Bronze ingestion complete — single PUT, no sidecar files')
print(f'   s3://{BRONZE}/{csv_key}   {len(csv_bytes)/1024/1024:.2f} MB')

# Prove the metadata really traveled with the object (no separate file to read)
head = s3.head_object(Bucket=BRONZE, Key=csv_key)
print('\n🔍 Metadata attached to the object (from head_object, not a .json file):')
for k, v in head['Metadata'].items():
    print(f'   {k:<16}: {v}')

---
## 2 · Silver — Clean, Validate, Partition

Silver applies **business rules and data contracts**:
- Remove duplicates
- Drop rows that violate physical constraints (price ≤ 0, dimensions = 0, carat < 0)
- Normalize inconsistent labels
- Add derived business columns (`price_tier`, `volume_mm3`)
- Cast to proper types to reduce memory footprint
- **Partition by `cut`** — physical query pruning for downstream analytics

Every rule is logged in a **Data Quality Report** that we'll print to the notebook and, in condensed form, attach as metadata on each Silver Parquet file we write.

The next cell only reads the raw CSV back from Bronze — the one and only place Bronze is read from, keeping it the immutable source of truth. We'll also peek at the object's metadata to see the provenance info that traveled with the file.

In [ ]:
# ── Read from Bronze ──────────────────────────────────────────────────────────
obj = s3.get_object(Bucket=BRONZE, Key=csv_key)
df_bronze = pd.read_csv(io.BytesIO(obj['Body'].read()))
print(f'📥 Read from Bronze: {len(df_bronze):,} rows × {len(df_bronze.columns)} columns')

# The ingestion metadata is still on the object — no metadata.json to look up
print(f'   source_url from object metadata: {obj["Metadata"].get("source_url")}')

### Applying the 7 Silver rules

The next cell walks through the Bronze data and applies, in order:

1. Drop exact duplicate rows.
2. Normalize `cut` labels (title-case) and drop anything that isn't a valid grade.
3. Drop rows with a null `price`.
4. Drop rows with zero-value dimensions (`x`, `y`, `z`).
5. Drop rows with a negative `carat`.
6. Cast numeric columns to smaller dtypes (`float32`/`int32`) and categorical columns to ordered `Categorical` — this shrinks memory and lets Parquet store better min/max statistics.
7. Add two derived columns: `volume_mm3` and `price_tier`.

Each step's row count is recorded into a `qr` (quality report) dict, which we print as a table and later embed as metadata on the Silver Parquet files.

In [ ]:
# ── Data Quality Report ───────────────────────────────────────────────────────
qr = {}  # quality report
df = df_bronze.copy()
start_rows = len(df)

# Rule 1: Remove exact duplicates
df = df.drop_duplicates()
qr['duplicates_removed'] = start_rows - len(df)

# Rule 2: Normalize cut labels (title-case)
CUT_ORDER = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
before = df['cut'].nunique()
df['cut'] = df['cut'].str.strip().str.title()
df = df[df['cut'].isin(CUT_ORDER)]
qr['invalid_cut_labels_removed'] = start_rows - qr['duplicates_removed'] - len(df)

# Rule 3: Drop null prices
before = len(df)
df = df.dropna(subset=['price'])
qr['null_price_rows_removed'] = before - len(df)

# Rule 4: Drop invalid dimensions
before = len(df)
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)]
qr['zero_dimension_rows_removed'] = before - len(df)

# Rule 5: Drop negative carats
before = len(df)
df = df[df['carat'] > 0]
qr['negative_carat_rows_removed'] = before - len(df)

# Rule 6: Cast to efficient types
df['price']   = df['price'].astype('int32')
df['carat']   = df['carat'].astype('float32')
df['depth']   = df['depth'].astype('float32')
df['table']   = df['table'].astype('float32')
df['x']       = df['x'].astype('float32')
df['y']       = df['y'].astype('float32')
df['z']       = df['z'].astype('float32')
df['cut']     = pd.Categorical(df['cut'], categories=CUT_ORDER, ordered=True)
df['color']   = pd.Categorical(df['color'], categories=list('DEFGHIJ'), ordered=True)
df['clarity'] = pd.Categorical(df['clarity'],
                               categories=['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF'],
                               ordered=True)

# Rule 7: Derived columns
df['volume_mm3']  = (df['x'] * df['y'] * df['z']).round(3).astype('float32')
df['price_tier']  = pd.cut(
    df['price'],
    bins=[0, 1000, 5000, 15000, df['price'].max() + 1],
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'],
    ordered=True,
)

final_rows = len(df)
qr['rows_in']    = start_rows
qr['rows_out']   = final_rows
qr['rows_total_removed'] = start_rows - final_rows
qr['pass_rate']  = f'{final_rows / start_rows * 100:.1f}%'

print('📋 Data Quality Report — Bronze → Silver')
print('─' * 45)
print(f'  Rows received (Bronze):    {qr["rows_in"]:>8,}')
print(f'  Duplicates removed:        {qr["duplicates_removed"]:>8,}')
print(f'  Invalid cut labels:        {qr["invalid_cut_labels_removed"]:>8,}')
print(f'  Null price rows:           {qr["null_price_rows_removed"]:>8,}')
print(f'  Zero dimension rows:       {qr["zero_dimension_rows_removed"]:>8,}')
print(f'  Negative carat rows:       {qr["negative_carat_rows_removed"]:>8,}')
print('─' * 45)
print(f'  Rows output (Silver):      {qr["rows_out"]:>8,}')
print(f'  Pass rate:                 {qr["pass_rate"]:>8}')

### Inspecting the resulting schema

The next cell just prints the Silver DataFrame's schema — dtype, non-null count, cardinality, and a sample value per column — so we can visually confirm the type casts and derived columns landed correctly before writing anything to S3.

In [ ]:
print('📐 Silver Schema (typed, enriched):')
schema_info = pd.DataFrame({
    'dtype':    df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'unique':   df.nunique(),
    'sample':   [str(df[c].iloc[0]) for c in df.columns],
})
display(schema_info)

### Partitioning + metadata, in the same PUT

The next cell writes one Parquet file per `cut` value (physical partitioning) using `s3_put_parquet`. For each partition we pass a `metadata` dict — partition name, row count, and a condensed version of the Bronze→Silver quality report — straight into the same `put_object` call the helper makes.

There's no standalone `_quality_report.json` file anymore: the report values live as S3 object metadata on the partitions themselves, retrievable at any time with `head_object`.

In [ ]:
# ── Partition by cut and upload to Silver (data + metadata in one PUT) ────────
print('📤 Writing Silver partitions to RustFS...')
print()

partition_sizes = {}

for cut_name in CUT_ORDER:
    partition_df = df[df['cut'] == cut_name].copy()
    key = f'{SILVER_PREFIX}/cut={cut_name}/data.parquet'

    partition_metadata = {
        'source_layer':               'bronze',
        'partition_cut':               cut_name,
        'rows':                        str(len(partition_df)),
        'rows_in_bronze':              str(qr['rows_in']),
        'rows_out_silver_total':       str(qr['rows_out']),
        'duplicates_removed':          str(qr['duplicates_removed']),
        'invalid_cut_labels_removed':  str(qr['invalid_cut_labels_removed']),
        'null_price_rows_removed':     str(qr['null_price_rows_removed']),
        'zero_dimension_rows_removed': str(qr['zero_dimension_rows_removed']),
        'negative_carat_rows_removed': str(qr['negative_carat_rows_removed']),
        'pass_rate':                   qr['pass_rate'],
        'schema_version':              '1.0',
    }

    size = s3_put_parquet(partition_df, SILVER, key, metadata=partition_metadata)
    partition_sizes[cut_name] = {'rows': len(partition_df), 'bytes': size}
    print(f'  s3://silver/{key}')
    print(f'    {len(partition_df):>6,} rows  |  {size/1024:.1f} KB  |  metadata attached in the same PUT')

total_silver_bytes = sum(p['bytes'] for p in partition_sizes.values())
bronze_size = len(csv_bytes)
print()
print(f'✅ Silver complete — {final_rows:,} rows across {len(CUT_ORDER)} partitions')
print(f'   Bronze CSV:   {bronze_size/1024/1024:.2f} MB')
print(f'   Silver total: {total_silver_bytes/1024:.0f} KB  '
      f'({total_silver_bytes/bronze_size*100:.0f}% of bronze size)')

---
## 3 · Gold — Business KPIs

Gold reads from Silver (all partitions) and produces **analytics-ready tables** consumed by BI dashboards and pricing algorithms.

We compute 4 KPI tables:

| Table | Business Question |
|-------|------------------|
| `price_by_cut_color` | Which cut × color combinations command premium prices? |
| `market_segments` | How is inventory distributed across price tiers? |
| `carat_price_bins` | What is the price curve as carat weight increases? |
| `quality_matrix` | Cut × Clarity heatmap — where is the value concentration? |

The next cell just reads every Silver `cut=*` partition back and concatenates them into one DataFrame — the only place Silver is read from in this section.

In [ ]:
# ── Read all Silver partitions ────────────────────────────────────────────────
print('📥 Reading all Silver partitions...')
partitions = []
for obj in s3.list_objects_v2(Bucket=SILVER, Prefix=f'{SILVER_PREFIX}/cut=')['Contents']:
    partitions.append(s3_read_parquet(SILVER, obj['Key']))
    print(f'   {obj["Key"]}  →  {len(partitions[-1]):,} rows')

df_silver = pd.concat(partitions, ignore_index=True)
# Restore ordered categoricals after concat
df_silver['cut']     = pd.Categorical(df_silver['cut'], categories=CUT_ORDER, ordered=True)
df_silver['color']   = pd.Categorical(df_silver['color'], categories=list('DEFGHIJ'), ordered=True)
df_silver['price_tier'] = pd.Categorical(df_silver['price_tier'],
                                         categories=['Budget','Mid-Range','Premium','Luxury'],
                                         ordered=True)
print(f'\n✅ Total Silver rows: {len(df_silver):,}')

### Computing KPIs and writing Gold, metadata included

The next cell computes the four KPI tables and writes each one to `s3://gold/kpi/` with `s3_put_parquet`. Just like in Silver, each call passes a `metadata` dict (source layer, KPI name, row count, generation timestamp) directly into the PUT — Gold has no manifest files either, every Parquet file is self-describing.

In [ ]:
# ── KPI 1: Price by Cut × Color ───────────────────────────────────────────────
generated_at = datetime.now(timezone.utc).isoformat()

price_by_cut_color = (
    df_silver.groupby(['cut', 'color'], observed=True)['price']
    .agg(avg_price='mean', median_price='median', count='count', max_price='max')
    .round(2)
    .reset_index()
)
s3_put_parquet(
    price_by_cut_color, GOLD, f'{GOLD_PREFIX}/price_by_cut_color.parquet',
    metadata={
        'source_layer': 'silver', 'kpi': 'price_by_cut_color',
        'rows': str(len(price_by_cut_color)), 'generated_at': generated_at,
    },
)
print('✅ KPI 1: price_by_cut_color —', len(price_by_cut_color), 'segments')
display(price_by_cut_color.head(7))

# ── KPI 2: Market Segments (Price Tier Distribution) ─────────────────────────
market_segments = (
    df_silver.groupby('price_tier', observed=True)
    .agg(
        stone_count=('price', 'count'),
        total_market_value=('price', 'sum'),
        avg_price=('price', 'mean'),
        avg_carat=('carat', 'mean'),
    )
    .round(2)
    .reset_index()
)
market_segments['market_share_pct'] = (
    market_segments['stone_count'] / market_segments['stone_count'].sum() * 100
).round(1)
s3_put_parquet(
    market_segments, GOLD, f'{GOLD_PREFIX}/market_segments.parquet',
    metadata={
        'source_layer': 'silver', 'kpi': 'market_segments',
        'rows': str(len(market_segments)), 'generated_at': generated_at,
    },
)
print('\n✅ KPI 2: market_segments —', len(market_segments), 'price tiers')
display(market_segments)

# ── KPI 3: Carat → Price Curve ────────────────────────────────────────────────
df_silver['carat_bin'] = pd.cut(df_silver['carat'],
                                bins=[0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0, 5.1],
                                labels=['0.2-0.4','0.4-0.6','0.6-0.8','0.8-1.0',
                                        '1.0-1.5','1.5-2.0','2.0-3.0','3.0+'])
carat_price_bins = (
    df_silver.groupby('carat_bin', observed=True)
    .agg(avg_price=('price','mean'), stone_count=('price','count'),
         avg_carat=('carat','mean'))
    .round(2).reset_index()
)
s3_put_parquet(
    carat_price_bins, GOLD, f'{GOLD_PREFIX}/carat_price_bins.parquet',
    metadata={
        'source_layer': 'silver', 'kpi': 'carat_price_bins',
        'rows': str(len(carat_price_bins)), 'generated_at': generated_at,
    },
)
print('\n✅ KPI 3: carat_price_bins —', len(carat_price_bins), 'bins')
display(carat_price_bins)

# ── KPI 4: Cut × Clarity Quality Matrix ──────────────────────────────────────
quality_matrix = (
    df_silver.groupby(['cut', 'clarity'], observed=True)['price']
    .mean().round(0).reset_index().rename(columns={'price': 'avg_price'})
)
s3_put_parquet(
    quality_matrix, GOLD, f'{GOLD_PREFIX}/quality_matrix.parquet',
    metadata={
        'source_layer': 'silver', 'kpi': 'quality_matrix',
        'rows': str(len(quality_matrix)), 'generated_at': generated_at,
    },
)
print('\n✅ KPI 4: quality_matrix —', len(quality_matrix), 'cut×clarity combinations')

---
## 4 · Analytics Dashboard

Read from Gold and produce the pricing intelligence dashboard.

The next cell will:

1. Read the 4 Gold Parquet tables back with `s3_read_parquet`.
2. Build a 2×2 `matplotlib` figure: a bar chart, a donut chart, a line chart, and a heatmap — one panel per KPI table.
3. Save the figure locally to `../temp/dashboard.png` and display it inline.

In [ ]:
# ── Load all Gold KPIs ────────────────────────────────────────────────────────
kpi1 = s3_read_parquet(GOLD, f'{GOLD_PREFIX}/price_by_cut_color.parquet')
kpi2 = s3_read_parquet(GOLD, f'{GOLD_PREFIX}/market_segments.parquet')
kpi3 = s3_read_parquet(GOLD, f'{GOLD_PREFIX}/carat_price_bins.parquet')
kpi4 = s3_read_parquet(GOLD, f'{GOLD_PREFIX}/quality_matrix.parquet')

# ── 2×2 Dashboard ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('DiamondDB Analytics — Pricing Intelligence Dashboard\n'
             'Gold Layer · DiamondDB Analytics Platform', fontsize=14, fontweight='bold', y=0.98)

PALETTE = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']

# ── Panel 1: Average Price by Cut (bar chart) ─────────────────────────────────
ax1 = axes[0, 0]
kpi1_cut = kpi1.groupby('cut')['avg_price'].mean().reset_index()
# Sort by CUT_ORDER
kpi1_cut['cut'] = pd.Categorical(kpi1_cut['cut'], categories=CUT_ORDER, ordered=True)
kpi1_cut = kpi1_cut.sort_values('cut')
bars = ax1.bar(kpi1_cut['cut'], kpi1_cut['avg_price'], color=PALETTE)
ax1.set_title('Avg Price by Cut Quality', fontweight='bold')
ax1.set_xlabel('Cut Grade')
ax1.set_ylabel('Average Price (USD)')
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, kpi1_cut['avg_price']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'${val:,.0f}', ha='center', fontsize=8.5, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, kpi1_cut['avg_price'].max() * 1.15)

# ── Panel 2: Market Segment Donut ─────────────────────────────────────────────
ax2 = axes[0, 1]
seg_colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']
wedges, texts, autotexts = ax2.pie(
    kpi2['stone_count'],
    labels=[f"{r['price_tier']}\n(${r['avg_price']:,.0f} avg)" for _, r in kpi2.iterrows()],
    autopct='%1.1f%%',
    colors=seg_colors,
    pctdistance=0.75,
    wedgeprops={'width': 0.5},  # donut hole
    startangle=90,
)
ax2.set_title('Market Segment Distribution\nby Stone Count', fontweight='bold')
total_value = kpi2['total_market_value'].sum()
ax2.text(0, 0, f'${total_value/1e6:.0f}M\nTotal GMV', ha='center', va='center',
         fontsize=9, fontweight='bold')

# ── Panel 3: Carat → Price Curve ─────────────────────────────────────────────
ax3 = axes[1, 0]
ax3.plot(kpi3['carat_bin'], kpi3['avg_price'], 'o-', color='#e74c3c', linewidth=2.5,
         markersize=8, markerfacecolor='white', markeredgewidth=2)
ax3.fill_between(range(len(kpi3)), kpi3['avg_price'], alpha=0.15, color='#e74c3c')
ax3.set_title('Price Curve by Carat Weight', fontweight='bold')
ax3.set_xlabel('Carat Range')
ax3.set_ylabel('Average Price (USD)')
ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3.set_xticks(range(len(kpi3)))
ax3.set_xticklabels(kpi3['carat_bin'], rotation=25, ha='right', fontsize=8)
ax3.grid(True, alpha=0.3)

# Annotate the 1-carat price premium
idx_1ct = kpi3[kpi3['carat_bin'] == '0.8-1.0'].index
if len(idx_1ct):
    val = kpi3.loc[idx_1ct[0], 'avg_price']
    ax3.annotate('1-carat\npremium', xy=(idx_1ct[0], val),
                 xytext=(idx_1ct[0] - 0.5, val * 1.25),
                 arrowprops=dict(arrowstyle='->', color='#e74c3c'),
                 fontsize=8, color='#e74c3c', fontweight='bold')

# ── Panel 4: Cut × Clarity Heatmap ───────────────────────────────────────────
ax4 = axes[1, 1]
CLARITY_ORDER = ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']
pivot = kpi4.pivot(index='cut', columns='clarity', values='avg_price')
pivot = pivot.reindex(index=CUT_ORDER, columns=CLARITY_ORDER)
im = ax4.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax4.set_xticks(range(len(CLARITY_ORDER)))
ax4.set_yticks(range(len(CUT_ORDER)))
ax4.set_xticklabels(CLARITY_ORDER)
ax4.set_yticklabels(CUT_ORDER)
ax4.set_title('Avg Price Heatmap\nCut × Clarity (green = highest value)', fontweight='bold')
ax4.set_xlabel('Clarity Grade  (→ better)')
ax4.set_ylabel('Cut Grade  (↓ better)')
for i in range(len(CUT_ORDER)):
    for j in range(len(CLARITY_ORDER)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax4.text(j, i, f'${val/1000:.1f}K', ha='center', va='center',
                     fontsize=7, color='black', fontweight='bold')
plt.colorbar(im, ax=ax4, shrink=0.8, label='Avg Price (USD)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('../temp/dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Dashboard rendered from Gold layer data')

---
## 5 · Lakehouse Inventory

List every object across all three layers — the full medallion footprint.

The next cell lists every key under each bucket's prefix. Notice there are **no `.json` sidecar files anywhere** — every object is either the raw CSV (Bronze) or a Parquet file (Silver/Gold), and every one of them carries its own metadata invisibly, retrievable with `head_object` if you need to inspect it.

In [ ]:
import os

LAYER_PREFIXES = {
    'bronze': 'diamonds/',
    'silver': 'diamonds/',
    'gold':   'kpi/',
}

print('🗂️  Lakehouse Inventory — All Layers\n')
total_bytes = 0
total_objects = 0

for bucket, prefix in LAYER_PREFIXES.items():
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = resp.get('Contents', [])
    layer_bytes = sum(o['Size'] for o in objects)
    total_bytes += layer_bytes
    total_objects += len(objects)
    print(f's3://{bucket}/{prefix}  ({len(objects)} objects · {layer_bytes/1024:.1f} KB)')
    for obj in sorted(objects, key=lambda o: o['Key']):
        fmt = os.path.splitext(obj['Key'])[1].upper().lstrip('.') or 'FILE'
        print(f'  └─ {obj["Key"]:<60}  {obj["Size"]/1024:>6.1f} KB  [{fmt}]')
    print()

print(f'Total: {total_objects} objects · {total_bytes/1024:.1f} KB across 3 layers')

---
## 6 · Format Efficiency Comparison

The next cell compares storage footprint across Bronze (CSV), Silver (partitioned Parquet), and Gold (aggregated Parquet) using `head_object` sizes, and plots a horizontal bar chart.

In [ ]:
# Bronze CSV vs Silver Parquet vs Gold Parquet
bronze_obj = s3.head_object(Bucket=BRONZE, Key=csv_key)
bronze_kb = bronze_obj['ContentLength'] / 1024

silver_kb = sum(
    s3.head_object(Bucket=SILVER, Key=obj['Key'])['ContentLength']
    for obj in s3.list_objects_v2(Bucket=SILVER, Prefix=f'{SILVER_PREFIX}/cut=')['Contents']
) / 1024

gold_kb = sum(
    s3.head_object(Bucket=GOLD, Key=obj['Key'])['ContentLength']
    for obj in s3.list_objects_v2(Bucket=GOLD, Prefix=f'{GOLD_PREFIX}/')['Contents']
) / 1024

fig, ax = plt.subplots(figsize=(8, 4))
labels = ['Bronze\n(CSV, 54K rows)', 'Silver\n(Parquet, 5 partitions)', 'Gold\n(Parquet, 4 KPI tables)']
sizes  = [bronze_kb, silver_kb, gold_kb]
colors = ['#cd6133', '#7f8c8d', '#f1c40f']

bars = ax.barh(labels, sizes, color=colors, edgecolor='white', height=0.5)
ax.set_xlabel('Storage Size (KB)')
ax.set_title('Storage Footprint Across Medallion Layers', fontweight='bold')
for bar, val in zip(bars, sizes):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f} KB', va='center', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Bronze CSV:       {bronze_kb:.0f} KB  (baseline)')
print(f'Silver Parquet:   {silver_kb:.0f} KB  ({silver_kb/bronze_kb*100:.0f}% of bronze — Parquet compression)')
print(f'Gold Parquet:     {gold_kb:.0f} KB  ({gold_kb/bronze_kb*100:.1f}% of bronze — aggregated KPIs only)')

---
## Cleanup

The next cell deletes every object we created in `bronze`, `silver`, and `gold`, plus the local dashboard PNG, so the lab leaves the environment as it found it.

In [ ]:
import os

for bucket, prefix in LAYER_PREFIXES.items():
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    for obj in resp.get('Contents', []):
        s3.delete_object(Bucket=bucket, Key=obj['Key'])
        print(f'🗑️  s3://{bucket}/{obj["Key"]}')

# Remove local dashboard export
try:
    os.remove('../temp/dashboard.png')
except FileNotFoundError:
    pass

print('\n✅ Cleanup complete')

---
## Summary

You built a **production-grade medallion data lakehouse** on RustFS:

| Layer | Key decisions | Output |
|-------|--------------|--------|
| **Bronze** | Store raw data exactly as received; zero transformation; metadata embedded in the same PUT (S3 object `Metadata`) | `raw.csv` only — no sidecar file |
| **Silver** | 7 quality rules; partition by `cut`; typed Parquet; metadata embedded per partition PUT | 5 partitions, ~70% size reduction, no manifest file |
| **Gold** | 4 KPI aggregations; read-optimized Parquet; metadata embedded per KPI PUT | 4 tables, <1% of bronze size, no manifest file |

### Design principles applied

**Immutable Bronze** — never edit raw data, and never transform it on the way in either. When your Silver logic has a bug, reprocess from Bronze. Without Bronze, a bad transform permanently destroys original data.

**Metadata travels with the object** — every PUT in this lab carried its own metadata (`Metadata=` on `put_object`). There's no separate `.json` file that can drift out of sync with its data file, no risk of writing the data but forgetting the sidecar (or vice versa), and provenance is always one `head_object` call away.

**Partition pruning** — by partitioning Silver by `cut`, a query like `WHERE cut = 'Ideal'` reads only 1/5 of the data. At petabyte scale this is the difference between seconds and hours.

**Typed schemas** — casting `price` to `int32` and floats to `float32` halves memory usage. Parquet stores min/max statistics per column per row group, enabling further skip pruning.

**Gold is purpose-built** — Gold tables are denormalized and pre-aggregated for specific queries. A BI dashboard should never hit Silver directly; Silver is for data engineers, Gold is for analysts.

### Next steps

- **Add Delta Lake / Apache Iceberg** for ACID transactions and time-travel on top of these Parquet files
- **Schedule ingestion** with Apache Airflow to run Bronze ingest daily
- **Query Gold directly** with DuckDB: `SELECT * FROM 's3://gold/kpi/*.parquet'`
- **Lab 05**: Add object versioning to track schema evolution across monthly ingestions
